In [6]:
import numpy as np
from PIL import Image
from typing import Dict, Tuple


def load_image_as_array(image_path: str) -> np.ndarray:
    """Загружает изображение и преобразует его в массив numpy в оттенках серого."""
    image = Image.open(image_path).convert('L')
    return np.array(image, dtype=np.int32)


def initialize_first_pixel(img_array: np.ndarray) -> Tuple[np.ndarray, int, Dict[int, int]]:
    """Инициализирует обработку первого пикселя изображения."""
    height, width = img_array.shape
    classes = np.zeros((height, width), dtype=int)
    class_counter = 1
    class_values = {class_counter: img_array[0, 0]}
    classes[0, 0] = class_counter
    class_counter += 1
    return classes, class_counter, class_values


def process_first_row(img_array: np.ndarray, classes: np.ndarray, 
                      class_counter: int, class_values: Dict[int, int], 
                      threshold: int) -> Tuple[np.ndarray, int, Dict[int, int]]:
    """Обрабатывает первую строку изображения."""
    width = img_array.shape[1]
    
    for j in range(1, width):
        current_pixel = img_array[0, j]
        left_pixel_class = classes[0, j-1]
        left_class_value = class_values[left_pixel_class]
        
        if abs(current_pixel - left_class_value) <= threshold:
            classes[0, j] = left_pixel_class
            # Обновляем среднее значение класса
            count = np.sum(classes == left_pixel_class)
            class_values[left_pixel_class] = (left_class_value * count + current_pixel) // (count + 1)
        else:
            classes[0, j] = class_counter
            class_values[class_counter] = current_pixel
            class_counter += 1
    
    return classes, class_counter, class_values


def process_first_column(img_array: np.ndarray, classes: np.ndarray, 
                         class_counter: int, class_values: Dict[int, int], 
                         threshold: int) -> Tuple[np.ndarray, int, Dict[int, int]]:
    """Обрабатывает первый столбец изображения (кроме первого пикселя)."""
    height = img_array.shape[0]
    
    for i in range(1, height):
        current_pixel = img_array[i, 0]
        upper_pixel_class = classes[i-1, 0]
        upper_class_value = class_values[upper_pixel_class]
        
        if abs(current_pixel - upper_class_value) <= threshold:
            classes[i, 0] = upper_pixel_class
            # Обновляем среднее значение класса
            count = np.sum(classes == upper_pixel_class)
            class_values[upper_pixel_class] = (upper_class_value * count + current_pixel) // (count + 1)
        else:
            classes[i, 0] = class_counter
            class_values[class_counter] = current_pixel
            class_counter += 1
    
    return classes, class_counter, class_values


def process_pixel(i: int, j: int, img_array: np.ndarray, classes: np.ndarray, 
                  class_values: Dict[int, int], threshold: int, delta: int) -> Tuple[np.ndarray, Dict[int, int]]:
    """Обрабатывает один пиксель изображения (кроме первой строки и первого столбца)."""
    current_pixel = img_array[i, j]
    left_pixel_class = classes[i, j-1]
    upper_pixel_class = classes[i-1, j]
    
    left_class_value = class_values[left_pixel_class]
    upper_class_value = class_values[upper_pixel_class]
    
    left_diff = abs(current_pixel - left_class_value)
    upper_diff = abs(current_pixel - upper_class_value)
    
    # Проверяем, принадлежат ли левый и верхний пиксели одному классу
    same_class = (left_pixel_class == upper_pixel_class)
    
    if left_diff > threshold and upper_diff > threshold:
        # Создаем новый класс
        classes[i, j] = max(class_values.keys()) + 1
        class_values[classes[i, j]] = current_pixel
    elif left_diff <= threshold and upper_diff > threshold:
        # Добавляем к левому классу
        classes[i, j] = left_pixel_class
        # Обновляем среднее значение класса
        count = np.sum(classes == left_pixel_class)
        class_values[left_pixel_class] = (left_class_value * count + current_pixel) // (count + 1)
    elif left_diff > threshold and upper_diff <= threshold:
        # Добавляем к верхнему классу
        classes[i, j] = upper_pixel_class
        # Обновляем среднее значение класса
        count = np.sum(classes == upper_pixel_class)
        class_values[upper_pixel_class] = (upper_class_value * count + current_pixel) // (count + 1)
    else:
        # Оба соседа подходят
        if same_class or abs(left_class_value - upper_class_value) <= delta:
            # Объединяем классы (если они разные)
            if not same_class:
                # Объединяем классы (переназначаем все пиксели верхнего класса к левому)
                classes[classes == upper_pixel_class] = left_pixel_class
                # Обновляем среднее значение объединенного класса
                count_left = np.sum(classes == left_pixel_class)
                new_value = (left_class_value * np.sum(classes == left_pixel_class) + 
                            upper_class_value * np.sum(classes == upper_pixel_class)) // count_left
                class_values[left_pixel_class] = new_value
                del class_values[upper_pixel_class]
            
            # Добавляем текущий пиксель к объединенному классу
            classes[i, j] = left_pixel_class
            # Обновляем среднее значение класса
            count = np.sum(classes == left_pixel_class)
            class_values[left_pixel_class] = (class_values[left_pixel_class] * (count - 1) + current_pixel) // count
        else:
            # Добавляем к классу с минимальным отклонением
            if left_diff <= upper_diff:
                classes[i, j] = left_pixel_class
                # Обновляем среднее значение класса
                count = np.sum(classes == left_pixel_class)
                class_values[left_pixel_class] = (left_class_value * count + current_pixel) // (count + 1)
            else:
                classes[i, j] = upper_pixel_class
                # Обновляем среднее значение класса
                count = np.sum(classes == upper_pixel_class)
                class_values[upper_pixel_class] = (upper_class_value * count + current_pixel) // (count + 1)
    
    return classes, class_values


def region_based_segmentation(image_path: str, threshold: int, delta: int) -> np.ndarray:
    """
    Выполняет сегментацию изображения на основе регионов.
    
    Args:
        image_path: Путь к входному изображению
        threshold: Пороговое значение для определения принадлежности к региону
        delta: Максимальная разница между средними значениями для объединения регионов
    
    Returns:
        Массив numpy с метками регионов
    """
    # Загрузка изображения
    img_array = load_image_as_array(image_path)
    height, width = img_array.shape
    
    # Инициализация матрицы классов (регионов)
    classes, class_counter, class_values = initialize_first_pixel(img_array)
    
    # Обработка первой строки
    classes, class_counter, class_values = process_first_row(
        img_array, classes, class_counter, class_values, threshold
    )
    
    # Обработка первого столбца
    classes, class_counter, class_values = process_first_column(
        img_array, classes, class_counter, class_values, threshold
    )
    
    # Обработка остальных пикселей
    for i in range(1, height):
        for j in range(1, width):
            classes, class_values = process_pixel(
                i, j, img_array, classes, class_values, threshold, delta
            )
    
    return classes

In [7]:
# Замените 'input_image.png' на путь к вашему изображению
classes = region_based_segmentation('origins/sk.jpg', 15, 20)

# Визуализация результата
normalized_classes = (classes * (255 / np.max(classes))).astype(np.uint8)
result_image = Image.fromarray(normalized_classes)
result_image.save("results/task_6/output_regions.jpg")
result_image.show()